# Issues
---

- For vertical levels, a sigma-level approximation is used instead of the true hybrid-level coordinate system. The pressure at each approximated level is calculated as

$$
p_{\mathrm{level}}=\sigma_{\mathrm{level}}p_{\mathrm{surface}}.
$$

- SINGV-RCM uses a latitude–longitude grid, while StormCast expects HRRR grid dimensions associated with a Lambert conformal grid. This notebook only performs bilinear interpolation onto the required 512 × 640 shape.
- SINGV-RCM pressure-level values beneath terrain are stored as zero. These zeros are retained during conversion.
- True surface pressure is not available in the collected SINGV-RCM file. Mean sea-level pressure, psl, is temporarily used as an approximation.
- StormCast refc is approximated from SINGV-RCM precipitation rate, pr, using a Marshall–Palmer relationship. It is not true composite radar reflectivity.
- The available SINGV-RCM data ends before the GFS_FX archive begins. A recent proxy timestamp is therefore used for GFS conditioning during this initial technical test.

# Setup
---

In [ ]:
# 0. Install and imports

!pip install -q earth2studio[stormcast]

import importlib
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import earth2studio.run as run

import utils_singv3 as ut
importlib.reload(ut)

from earth2studio.models.px import StormCast
from earth2studio.data import GFS_FX
from earth2studio.io import ZarrBackend

from google.colab import drive

In [ ]:
# 1. Configuration

# Use 2 for the first smoke test.
# Change back to 12 after the complete pipeline works.
N_STEPS = 2

# One collected SingV3 file containing both surface and pressure variables
PATH_SINGV = "/content/singv_raw_20141201_0100.nc"

PATH_OUTPUT = f"stormcast_output_singv3_{N_STEPS}h.zarr"

# The actual SingV3 time is from 2014, which GFS_FX cannot retrieve.
# This newer time is used only to obtain GFS conditioning and run StormCast.
STORMCAST_TIME = np.datetime64(
    "2025-11-27T06:00:00",
    "ns",
)

drive.mount("/content/drive")

print("SingV3 file:", PATH_SINGV)
print("StormCast proxy time:", STORMCAST_TIME)
print("Output:", PATH_OUTPUT)

In [ ]:
# 2. Model and coordinate setup
package = StormCast.load_default_package()

model = StormCast.load_model(
    package,
    conditioning_data_source=GFS_FX(),
)

coords = model.input_coords()

VARIABLES = list(np.asarray(coords["variable"]).astype(str))
HRRR_Y = np.asarray(coords["hrrr_y"])
HRRR_X = np.asarray(coords["hrrr_x"])

NY = len(HRRR_Y)
NX = len(HRRR_X)

print("Number of variables:", len(VARIABLES))
print("Target grid:", NY, "×", NX)
print("First variables:", VARIABLES[:10])

Expected:

Number of variables: 99
Target grid: 512 × 640

In [ ]:
3. Helper function to build the StormCast input
def build_stormcast_input():
    """
    Convert one collected SingV3 file into the full StormCast input array.

    Conversion array:
        (variable, hrrr_y, hrrr_x)

    Final Earth2Studio array:
        (time, variable, hrrr_y, hrrr_x)
    """

    with xr.open_dataset(
        PATH_SINGV,
        mask_and_scale=True,
    ) as ds_singv:

        # Validate the original collected SingV3 file
        ut.validate_singv_dataset(
            ds_singv,
            verbose=True,
        )

        # Store the actual SingV3 valid time for reference
        singv_time = np.datetime64(
            str(ds_singv["valid_time"].values),
            "ns",
        )

        # Define the 512 × 640 target grid
        grid = ut.make_grid_spec(
            ds_surface=ds_singv,
            hrrr_y=HRRR_Y,
            hrrr_x=HRRR_X,
        )

        # Initially construct the data without a time dimension
        fields = ut.make_empty_field_array(
            variables=VARIABLES,
            hrrr_y=HRRR_Y,
            hrrr_x=HRRR_X,
        )

        # Surface variables:
        # tas → t2m
        # uas → u10m
        # vas → v10m
        # psl → mslp
        ut.fill_surface_fields(
            data=fields,
            ds_surface=ds_singv,
            grid=grid,
            verbose=True,
        )

        # First-pass approximation:
        # surface pressure ≈ mean sea-level pressure
        surface_pressure = ut.get_surface_pressure(
            ds_surface=ds_singv,
            grid=grid,
            verbose=True,
        )

        # Pressure levels → approximated HRRR hybrid levels
        ut.fill_hybrid_fields(
            data=fields,
            ds_pressure=ds_singv,
            sp=surface_pressure,
            grid=grid,
            verbose=True,
        )

        # pr → approximate refc
        ut.fill_refc(
            data=fields,
            ds_surface=ds_singv,
            grid=grid,
            verbose=True,
        )

        # Validate before introducing the time dimension
        ut.validate_field_array(
            fields=fields,
            variables=VARIABLES,
            ny=NY,
            nx=NX,
        )

    # Earth2Studio requires a time dimension.
    #
    # The assigned time is the proxy GFS time rather than the actual
    # historical SingV3 time.
    data = ut.add_time_dimension(
        data=fields,
        time_value=STORMCAST_TIME,
    )

    data.attrs["singv_valid_time"] = str(singv_time)
    data.attrs["stormcast_time"] = str(STORMCAST_TIME)
    data.attrs["time_warning"] = (
        "The SingV3 initial state and GFS conditioning are from "
        "different dates. This run is a technical smoke test."
    )

    return (
        data,
        grid.target_lats,
        grid.target_lons,
        singv_time,
    )

# Execution
---

In [ ]:
4. Build input array
print("Building StormCast input from SingV3...")

data, target_lats, target_lons, SINGV_TIME = (
    build_stormcast_input()
)

# This is the timestamp passed to Earth2Studio and GFS_FX.
STARTING_TIME = STORMCAST_TIME

print("\nActual SingV3 valid time:", SINGV_TIME)
print("StormCast/GFS time:", STARTING_TIME)

print("\nValidating final StormCast input array...")

ut.validate_stormcast_input_array(
    data,
    VARIABLES,
    NY,
    NX,
)

print("\nFinal input dimensions:", data.dims)
print("Final input shape:", data.shape)
print("Final input dtype:", data.dtype)

Expected:

Final input dimensions:
('time', 'variable', 'hrrr_y', 'hrrr_x')

Final input shape:
(1, 99, 512, 640)

In [ ]:
# 5. Inspect selected input fields

INPUT_VARS = [
    "t2m",
    "u10m",
    "v10m",
    "mslp",
    "t5hl",
    "q5hl",
    "Z5hl",
    "refc",
]

fig, axes = plt.subplots(
    2,
    4,
    figsize=(18, 9),
)

for ax, var in zip(axes.flat, INPUT_VARS):
    field = (
        data
        .sel(time=STARTING_TIME, variable=var)
        .values
    )

    if var == "refc":
        image = ax.imshow(
            field,
            origin="upper",
            cmap="Spectral_r",
            vmin=-10,
            vmax=60,
        )
    else:
        image = ax.imshow(
            field,
            origin="upper",
        )

    ax.set_title(
        f"{var}\n"
        f"min={field.min():.3g}, "
        f"mean={field.mean():.3g}, "
        f"max={field.max():.3g}"
    )

    ax.set_xlabel("hrrr_x")
    ax.set_ylabel("hrrr_y")

    plt.colorbar(
        image,
        ax=ax,
        shrink=0.8,
        label=ut.infer_unit(var),
    )

plt.suptitle(
    f"Converted SingV3 input fields\n"
    f"actual valid time: {SINGV_TIME}",
    fontsize=15,
)

plt.tight_layout()
plt.show()

In [ ]:
6. Wrap and sanity-check
my_data = ut.MyLocalData(data)

sample = my_data(
    [STARTING_TIME],
    VARIABLES,
)

print(
    "Earth2Studio wrapper verified:",
    sample.dims,
    sample.shape,
)

assert sample.shape == (1, 99, NY, NX)
assert np.isfinite(sample.values).all()

print("All wrapper values are finite.")

In [ ]:
7. Optionally save the prepared input

This allows the converted input to be reused without repeating the interpolation.

PATH_INPUT = "stormcast_input_singv3.nc"

data.to_netcdf(PATH_INPUT)

print("Saved prepared input:", PATH_INPUT)

It can later be reopened using:

# data = xr.open_dataarray(PATH_INPUT)
# my_data = ut.MyLocalData(data)

In [ ]:
8. Inference
io = ZarrBackend(
    PATH_OUTPUT,
    backend_kwargs={"overwrite": True},
)

io = run.deterministic(
    time=[STARTING_TIME],
    nsteps=N_STEPS,
    prognostic=model,
    data=my_data,
    io=io,
)

# Basic output inspection
---

In [ ]:
# 10. Open forecast output
ds_out = xr.open_zarr(PATH_OUTPUT)

ds_out
print("Output dimensions:")
print(ds_out.sizes)

print("\nLead times:")
print(ds_out["lead_time"].values)

expected_leads = N_STEPS + 1
actual_leads = ds_out.sizes["lead_time"]

if actual_leads != expected_leads:
    raise ValueError(
        f"Expected {expected_leads} lead times, "
        f"got {actual_leads}"
    )

In [ ]:
# 11. Check that lead zero reproduces the supplied input
LEAD0_VARS = [
    "t2m",
    "mslp",
    "u10m",
    "v10m",
    "t5hl",
    "q5hl",
    "Z5hl",
    "refc",
]

lead0_records = []

for var in LEAD0_VARS:
    input_field = (
        data
        .sel(time=STARTING_TIME, variable=var)
        .values
    )

    output_field = (
        ds_out[var]
        .isel(time=0, lead_time=0)
        .values
    )

    error = output_field - input_field

    rmse = float(
        np.sqrt(np.mean(error ** 2))
    )

    max_absolute_error = float(
        np.max(np.abs(error))
    )

    lead0_records.append(
        {
            "variable": var,
            "rmse": rmse,
            "max_absolute_error": max_absolute_error,
        }
    )

df_lead0 = pd.DataFrame(lead0_records)

df_lead0

Lead-zero errors should be zero or extremely small.

In [ ]:
# 12. Plot forecast evolution
PLOT_VARS = [
    "t2m",
    "mslp",
    "refc",
]

LEADS_TO_PLOT = sorted(
    set([
        0,
        1,
        N_STEPS // 2,
        N_STEPS,
    ])
)

for var in PLOT_VARS:
    fig, axes = plt.subplots(
        1,
        len(LEADS_TO_PLOT),
        figsize=(5 * len(LEADS_TO_PLOT), 4.5),
        squeeze=False,
    )

    for column, lead_index in enumerate(LEADS_TO_PLOT):
        ax = axes[0, column]

        field = (
            ds_out[var]
            .isel(time=0, lead_time=lead_index)
            .values
        )

        lead_hour = int(
            ds_out["lead_time"].values[lead_index]
            / np.timedelta64(1, "h")
        )

        if var == "refc":
            image = ax.imshow(
                field,
                origin="upper",
                cmap="Spectral_r",
                vmin=-10,
                vmax=60,
            )
        else:
            image = ax.imshow(
                field,
                origin="upper",
            )

        ax.set_title(
            f"{var}, lead {lead_hour} h"
        )

        ax.set_xlabel("hrrr_x")
        ax.set_ylabel("hrrr_y")

        plt.colorbar(
            image,
            ax=ax,
            shrink=0.8,
            label=ut.infer_unit(var),
        )

    fig.suptitle(
        f"StormCast forecast from SingV3 initial fields\n"
        f"SingV3 state: {SINGV_TIME}; "
        f"proxy GFS time: {STARTING_TIME}",
        fontsize=13,
    )

    plt.tight_layout()
    plt.show()